# Safety Gear Detection - Model Execution and Interpretation

This notebook provides tools for executing safety gear detection models and interpreting their results. You can use it to train, evaluate, and test different model architectures on safety gear detection tasks.

## 1. Setup and Imports

In [ ]:
import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

# Add project root to path to enable imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    
# Import project modules
from config import CFG
from detector import SafetyGearDetector
from data.dataset import create_data_loaders, create_yolo_data_loaders, create_detr_data_loaders
from utils.visualization import visualize_detections, plot_precision_recall_curve, visualize_yolo_results
from utils.evaluation import calculate_map, calculate_coco_map

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Set up paths
DATA_PATH = os.path.join(project_root, 'css-data')  # Update this to your dataset path
OUTPUT_DIR = os.path.join(project_root, 'output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Model parameters
MODEL_TYPE = 'faster_rcnn'  # Options: 'rcnn', 'fast_rcnn', 'faster_rcnn', 'mask_rcnn', 'yolov8', 'detr', 'rtdetr-l'
BATCH_SIZE = 4
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0005

# Inference parameters
CONFIDENCE_THRESHOLD = 0.5
NMS_THRESHOLD = 0.45

# Display class names
print(f"Classes: {CFG.CLASS_NAMES}")

## 3. Dataset Exploration

In [ ]:
# Helper function to visualize dataset samples
def visualize_dataset_samples(dataloader, class_names, num_samples=3):
    """Visualize samples from the dataset with annotations."""
    # Get a batch from the dataloader
    images, targets = next(iter(dataloader))
    
    for i in range(min(num_samples, len(images))):
        image = images[i]
        boxes = targets[i]['boxes'].numpy()
        labels = targets[i]['labels'].numpy()
        
        # Convert tensor to PIL Image for visualization
        if isinstance(image, torch.Tensor):
            # Denormalize if the image is normalized
            mean = torch.tensor([0.485, 0.456, 0.406])
            std = torch.tensor([0.229, 0.224, 0.225])
            image = image.permute(1, 2, 0) * std + mean
            image = image.numpy()
            image = (image * 255).astype(np.uint8)
        
        # Plot the image with bounding boxes
        plt.figure(figsize=(10, 8))
        plt.imshow(image)
        
        # Draw bounding boxes
        for box, label in zip(boxes, labels):
            x1, y1, x2, y2 = box
            rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=2)
            plt.gca().add_patch(rect)
            plt.text(x1, y1, f'{class_names[label]}', bbox=dict(facecolor='yellow', alpha=0.5))
        
        plt.axis('off')
        plt.tight_layout()
        plt.show()

In [ ]:
# Load data loaders based on model type
if 'yolo' in MODEL_TYPE.lower():
    # For YOLO models, create the data YAML
    train_loader, valid_loader, test_loader = create_yolo_data_loaders(
        train_dir=os.path.join(DATA_PATH, 'train'),
        valid_dir=os.path.join(DATA_PATH, 'val'),
        test_dir=os.path.join(DATA_PATH, 'test'),
        batch_size=BATCH_SIZE,
        img_size=640
    )
    print("Created YOLO data configuration")
    
    # Since YOLO handles data loading internally, we'll create a standard loader just for visualization
    vis_train_loader, _, _ = create_data_loaders(
        train_dir=os.path.join(DATA_PATH, 'train'),
        batch_size=BATCH_SIZE
    )
    
    # Visualize some training samples
    if vis_train_loader:
        print("\nTraining data samples:")
        visualize_dataset_samples(vis_train_loader, CFG.CLASS_NAMES)
        
elif 'detr' in MODEL_TYPE.lower():
    # For DETR models
    train_loader, valid_loader, test_loader = create_detr_data_loaders(
        data_dir=os.path.join(project_root, 'detr-data'),
        batch_size=BATCH_SIZE,
        train_dir=os.path.join(project_root, 'detr-data', 'train'),
        valid_dir=os.path.join(project_root, 'detr-data', 'valid'),
        test_dir=os.path.join(project_root, 'detr-data', 'test')
    )
    
    # Visualize some training samples
    if train_loader:
        print("\nTraining data samples (DETR format):")
        # DETR has a different data format, so we need a different visualization function
        # For simplicity, we'll skip visualization here
        print("DETR dataset loaded")
        
else:
    # Standard loaders for RCNN models
    train_loader, valid_loader, test_loader = create_data_loaders(
        train_dir=os.path.join(DATA_PATH, 'train'),
        valid_dir=os.path.join(DATA_PATH, 'val'),
        test_dir=os.path.join(DATA_PATH, 'test'),
        batch_size=BATCH_SIZE
    )
    
    # Visualize some training samples
    if train_loader:
        print("\nTraining data samples:")
        visualize_dataset_samples(train_loader, CFG.CLASS_NAMES)

## 4. Model Initialization

In [ ]:
# Initialize the detector with the specified model type
detector = SafetyGearDetector(model_type=MODEL_TYPE)
print(f"Initialized {MODEL_TYPE} model")

# Check if a pre-trained model exists and load it
model_path = os.path.join(OUTPUT_DIR, f"{MODEL_TYPE}_safety_gear.pt")
if os.path.exists(model_path):
    print(f"Loading pre-trained model from {model_path}")
    detector.load_model(model_path)
else:
    print("No pre-trained model found. Will need to train from scratch.")

## 5. Model Training (Optional)

This section allows you to train the model. You can skip this if you have a pre-trained model.

In [ ]:
# Set this to True to train the model or False to skip training
DO_TRAINING = True

if DO_TRAINING:
    print(f"Training {MODEL_TYPE} model for {NUM_EPOCHS} epochs...")
    
    # Training arguments
    training_args = {
        'epochs': NUM_EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'gradient_accumulation_steps': 1
    }
    
    # Print model configuration details
    if hasattr(detector.model, 'print_hyperparameters'):
        detector.model.print_hyperparameters(training_args)
    
    # Train the model
    history = detector.train(
        train_loader,
        valid_loader,
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )
    
    # Save the trained model
    model_save_path = os.path.join(OUTPUT_DIR, f"{MODEL_TYPE}_safety_gear.pt")
    detector.save_model(model_save_path)
    print(f"Model saved to {model_save_path}")
    
    # Plot training history if available
    if history and 'train_loss' in history:
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 2, 1)
        plt.plot(history['train_loss'])
        if 'val_loss' in history and history['val_loss']:
            plt.plot(history['val_loss'])
            plt.legend(['train_loss', 'val_loss'])
        else:
            plt.legend(['train_loss'])
        plt.title('Loss')
        plt.xlabel('Epoch')
        
        if 'val_map' in history and history['val_map']:
            plt.subplot(1, 2, 2)
            plt.plot(history['val_map'])
            plt.title('Validation mAP')
            plt.xlabel('Epoch')
        
        plt.tight_layout()
        plt.show()
        
    # For YOLO models, visualize using the dedicated function
    if 'yolo' in MODEL_TYPE.lower() and hasattr(history, 'maps'):
        visualize_yolo_results(history)
else:
    print("Skipping training phase")

## 6. Model Evaluation

In [ ]:
if test_loader:
    print("Evaluating model on test dataset...")
    metrics = detector.evaluate(test_loader)
    
    # Print main metrics
    print(f"\nEvaluation Results:")
    print(f"mAP@0.5: {metrics.get('mAP50', metrics.get('mAP', 0)):.4f}")
    if 'mAP_range' in metrics:
        print(f"mAP@0.5:0.95: {metrics['mAP_range']:.4f}")
    
    # Print per-class results if available
    if 'per_class_ap' in metrics:
        print("\nPer-class Average Precision:")
        for cls_name, ap in zip(CFG.CLASS_NAMES, metrics['per_class_ap']):
            print(f"  {cls_name}: {ap:.4f}")
    
    # Calculate and display COCO-style metrics (for RCNN models)
    if not 'yolo' in MODEL_TYPE.lower() and not 'detr' in MODEL_TYPE.lower():
        try:
            print("\nCalculating COCO-style metrics...")
            calculate_coco_map(detector, test_loader)
        except Exception as e:
            print(f"Error calculating COCO metrics: {e}")
    
    # Create confusion matrix if available
    if 'confusion_matrix' in metrics:
        plt.figure(figsize=(10, 8))
        sns.heatmap(
            metrics['confusion_matrix'], 
            annot=True, 
            fmt='d', 
            xticklabels=CFG.CLASS_NAMES,
            yticklabels=CFG.CLASS_NAMES
        )
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title('Confusion Matrix')
        plt.tight_layout()
        plt.show()
else:
    print("No test loader available. Skipping evaluation.")

## 7. Inference on Sample Images

In [ ]:
# Find test images
test_img_dir = os.path.join(DATA_PATH, 'test', 'images')
if not os.path.exists(test_img_dir):
    print(f"Test image directory not found at {test_img_dir}")
    print("Please provide a path to a directory with test images:")
    test_img_dir = input()

if os.path.exists(test_img_dir):
    test_images = [os.path.join(test_img_dir, f) for f in os.listdir(test_img_dir)
                  if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    # Select a few images for inference
    num_images = min(5, len(test_images))
    sample_images = test_images[:num_images]
    
    print(f"Running inference on {num_images} sample images...")
    
    # Process each image
    for img_path in sample_images:
        print(f"\nImage: {os.path.basename(img_path)}")
        
        # Run inference
        boxes, labels, scores, class_names = detector.predict(
            img_path, 
            confidence_threshold=CONFIDENCE_THRESHOLD,
            nms_threshold=NMS_THRESHOLD
        )
        
        # Print detection results
        print(f"Found {len(boxes)} detections:")
        for i, (box, label, score) in enumerate(zip(boxes, labels, scores)):
            class_name = class_names[label] if label < len(class_names) else f"Class {label}"
            print(f"  {i+1}. {class_name}: {score:.4f}, Box: {box}")
        
        # Visualize the predictions
        result_image = detector.visualize_prediction(img_path, boxes, labels, scores)
        
        # Display the result
        plt.figure(figsize=(12, 8))
        plt.imshow(result_image)
        plt.axis('off')
        plt.title(f"Detections on {os.path.basename(img_path)}")
        plt.tight_layout()
        plt.show()
        
        # Save the result
        output_path = os.path.join(OUTPUT_DIR, f"detection_{os.path.basename(img_path)}")
        plt.imsave(output_path, result_image)
        print(f"Saved result to {output_path}")
else:
    print("No test images found. Skipping inference.")

## 8. Model Performance Analysis

In [ ]:
# Generate precision-recall curves
if test_loader and not 'yolo' in MODEL_TYPE.lower():
    try:
        print("Generating precision-recall curves...")
        
        # Run inference on test data to collect predictions and ground truth
        all_predictions = []
        all_targets = []
        
        for images, targets in tqdm(test_loader, desc="Collecting predictions"):
            # Process each image in the batch
            for img, target in zip(images, targets):
                # Get predictions
                if isinstance(img, torch.Tensor):
                    # Need to convert single image to batch format for some models
                    img_batch = img.unsqueeze(0).to(device)
                    with torch.no_grad():
                        outputs = detector.model.model(img_batch)
                    output = outputs[0]  # Get the first (only) output
                else:
                    # For PIL images or other formats
                    boxes, labels, scores, _ = detector.predict(
                        img, 
                        confidence_threshold=0.0,  # Use low threshold to get all predictions
                        nms_threshold=NMS_THRESHOLD
                    )
                    output = {
                        'boxes': boxes,
                        'labels': labels,
                        'scores': scores
                    }
                
                all_predictions.append(output)
                all_targets.append(target)
        
        # Calculate mAP and other metrics
        results = calculate_map(
            all_predictions, 
            all_targets, 
            iou_threshold=0.5, 
            num_classes=len(CFG.CLASS_NAMES)
        )
        
        # Plot precision-recall curves for each class
        fig, axes = plt.subplots(1, len(CFG.CLASS_NAMES), figsize=(15, 5))
        if len(CFG.CLASS_NAMES) == 1:
            axes = [axes]  # Handle the case of only one class
            
        for i, (ax, class_name) in enumerate(zip(axes, CFG.CLASS_NAMES)):
            if 'precisions' in results and i < len(results['precisions']):
                precision = results['precisions'][i]
                recall = results['recalls'][i]
                ap = results['ap'][i]
                
                ax.plot(recall, precision)
                ax.set_title(f"{class_name}\nAP: {ap:.4f}")
                ax.set_xlabel('Recall')
                ax.set_ylabel('Precision')
                ax.set_xlim([0, 1])
                ax.set_ylim([0, 1.05])
        
        plt.tight_layout()
        plt.show()
        
        # Summary of results
        print(f"\nOverall mAP@0.5: {results['map']:.4f}")
        print("\nPer-class Average Precision:")
        for cls_name, ap in zip(CFG.CLASS_NAMES, results['ap']):
            print(f"  {cls_name}: {ap:.4f}")
            
    except Exception as e:
        print(f"Error generating precision-recall curves: {e}")
        import traceback
        traceback.print_exc()

## 9. Advanced Analysis: Error Cases

In [ ]:
# Find error cases (false positives, false negatives)
if test_loader:
    try:
        print("Finding and analyzing error cases...")
        
        # Set confidence threshold
        conf_threshold = 0.5
        
        # Find examples with high confidence false positives or missed detections
        false_positives = []
        false_negatives = []
        correct_detections = []
        
        for img_batch, targets in tqdm(test_loader, desc="Analyzing error cases"):
            for i, (img, target) in enumerate(zip(img_batch, targets)):
                if isinstance(img, torch.Tensor):
                    # Convert tensor to PIL or numpy for visualization
                    img_np = img.permute(1, 2, 0).numpy()
                    # Denormalize if needed
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img_np = img_np * std + mean
                    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
                else:
                    img_np = np.array(img)
                
                # Run detection
                with torch.no_grad():
                    if isinstance(img, torch.Tensor):
                        img_batch_single = img.unsqueeze(0).to(device)
                        outputs = detector.model.model(img_batch_single)
                        output = outputs[0]
                    else:
                        # For PIL images or other formats
                        boxes, labels, scores, _ = detector.predict(
                            img, 
                            confidence_threshold=0.0,  # Use low threshold 
                            nms_threshold=NMS_THRESHOLD
                        )
                        output = {
                            'boxes': boxes,
                            'labels': labels,
                            'scores': scores
                        }
                
                # Get ground truth
                gt_boxes = target['boxes'].numpy()
                gt_labels = target['labels'].numpy()
                
                # Get predictions with confidence > threshold
                pred_boxes = output['boxes']
                pred_scores = output['scores']
                pred_labels = output['labels']
                
                if isinstance(pred_boxes, torch.Tensor):
                    pred_boxes = pred_boxes.cpu().numpy()
                    pred_scores = pred_scores.cpu().numpy()
                    pred_labels = pred_labels.cpu().numpy()
                
                # Filter by confidence
                mask = pred_scores > conf_threshold
                high_conf_boxes = pred_boxes[mask]
                high_conf_scores = pred_scores[mask]
                high_conf_labels = pred_labels[mask]
                
                # Find matched detections using IoU
                matched_gt = set()
                matched_pred = set()
                
                # For each prediction, find the best matching ground truth
                for p_idx, (pred_box, pred_label, pred_score) in enumerate(zip(high_conf_boxes, high_conf_labels, high_conf_scores)):
                    best_iou = 0.5  # IoU threshold
                    best_gt_idx = -1
                    
                    for gt_idx, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
                        if gt_idx in matched_gt or gt_label != pred_label:
                            continue
                        
                        # Calculate IoU
                        x1 = max(pred_box[0], gt_box[0])
                        y1 = max(pred_box[1], gt_box[1])
                        x2 = min(pred_box[2], gt_box[2])
                        y2 = min(pred_box[3], gt_box[3])
                        
                        intersection = max(0, x2 - x1) * max(0, y2 - y1)
                        pred_area = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
                        gt_area = (gt_box[2] - gt_box[0]) * (gt_box[3] - gt_box[1])
                        union = pred_area + gt_area - intersection
                        
                        iou = intersection / union if union > 0 else 0
                        
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx
                    
                    if best_gt_idx >= 0:
                        # Found a matching ground truth
                        matched_gt.add(best_gt_idx)
                        matched_pred.add(p_idx)
                        
                        # Store as correct detection
                        correct_detections.append({
                            'image': img_np,
                            'pred_box': pred_box,
                            'gt_box': gt_boxes[best_gt_idx],
                            'label': pred_label,
                            'score': pred_score,
                            'iou': best_iou
                        })
                    else:
                        # False positive
                        false_positives.append({
                            'image': img_np,
                            'pred_box': pred_box,
                            'label': pred_label,
                            'score': pred_score
                        })
                
                # Find false negatives (ground truths without matches)
                for gt_idx, (gt_box, gt_label) in enumerate(zip(gt_boxes, gt_labels)):
                    if gt_idx not in matched_gt:
                        # False negative
                        false_negatives.append({
                            'image': img_np,
                            'gt_box': gt_box,
                            'label': gt_label
                        })
        
        # Display some false positives
        if false_positives:
            print(f"\nFound {len(false_positives)} false positives. Displaying a few:")
            num_to_show = min(3, len(false_positives))
            
            for i in range(num_to_show):
                fp = false_positives[i]
                plt.figure(figsize=(8, 6))
                plt.imshow(fp['image'])
                
                # Draw the false positive box
                box = fp['box'] if 'box' in fp else fp['pred_box']
                x1, y1, x2, y2 = box
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='red', linewidth=2)
                plt.gca().add_patch(rect)
                
                # Add label
                label = CFG.CLASS_NAMES[fp['label']] if fp['label'] < len(CFG.CLASS_NAMES) else f"Class {fp['label']}"
                plt.title(f"False Positive: {label}, Score: {fp['score']:.4f}")
                plt.axis('off')
                plt.tight_layout()
                plt.show()
        else:
            print("No false positives found.")
        
        # Display some false negatives
        if false_negatives:
            print(f"\nFound {len(false_negatives)} false negatives. Displaying a few:")
            num_to_show = min(3, len(false_negatives))
            
            for i in range(num_to_show):
                fn = false_negatives[i]
                plt.figure(figsize=(8, 6))
                plt.imshow(fn['image'])
                
                # Draw the missed ground truth box
                x1, y1, x2, y2 = fn['gt_box']
                rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor='blue', linewidth=2)
                plt.gca().add_patch(rect)
                
                # Add label
                label = CFG.CLASS_NAMES[fn['label']] if fn['label'] < len(CFG.CLASS_NAMES) else f"Class {fn['label']}"
                plt.title(f"Missed Detection: {label}")
                plt.axis('off')
                plt.tight_layout()
                plt.show()
        else:
            print("No false negatives found.")
        
    except Exception as e:
        print(f"Error analyzing error cases: {e}")
        import traceback
        traceback.print_exc()

## 10. Conclusion and Performance Summary

In [ ]:
# Print overall performance summary
print("\n" + "=" * 50)
print(f"MODEL PERFORMANCE SUMMARY: {MODEL_TYPE}")
print("=" * 50)

print(f"Model Type: {MODEL_TYPE}")
print(f"Classes: {CFG.CLASS_NAMES}")

if 'metrics' in locals():
    print(f"\nTest Set Performance:")
    print(f"mAP@0.5: {metrics.get('mAP50', metrics.get('mAP', 0)):.4f}")
    if 'mAP_range' in metrics:
        print(f"mAP@0.5:0.95: {metrics['mAP_range']:.4f}")
    
    # Other metrics that might be available
    for metric_name in ['precision', 'recall', 'f1']:
        if metric_name in metrics:
            print(f"{metric_name.capitalize()}: {metrics[metric_name]:.4f}")
    
    # Per-class metrics if available
    if 'per_class_ap' in metrics and len(metrics['per_class_ap']) > 0:
        print("\nPer-class AP:")
        for cls_name, ap in zip(CFG.CLASS_NAMES, metrics['per_class_ap']):
            print(f"  {cls_name}: {ap:.4f}")

# Model size
if hasattr(detector.model, 'model'):
    model_size_mb = sum(p.numel() * p.element_size() for p in detector.model.model.parameters()) / (1024 * 1024)
    print(f"\nModel Size: {model_size_mb:.2f} MB")
    
    # Count parameters
    total_params = sum(p.numel() for p in detector.model.model.parameters())
    trainable_params = sum(p.numel() for p in detector.model.model.parameters() if p.requires_grad)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,} ({trainable_params/total_params:.2%})")

print("\nConclusion:")
print("This notebook demonstrated the complete workflow for safety gear detection using")
print(f"the {MODEL_TYPE} model, including training, evaluation, and inference.")
print("You can experiment with different model architectures and hyperparameters")
print("to improve performance on your specific safety gear detection tasks.")